# 05 — LSTM Sequence Modelling

**LSTM means Long Short-Term Memory.** It is a recurrent neural-network architecture designed to carry useful information across a sequence while learning what to keep, update and forget. This notebook uses synthetic sequences so the tensor shapes and training loop stay easy to inspect.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
samples, steps, features = 1600, 20, 1
X = torch.randn(samples, steps, features)
# Class 1 when the second half of the sequence has the larger total signal.
first_half = X[:, :10, 0].sum(dim=1)
second_half = X[:, 10:, 0].sum(dim=1)
y = (second_half > first_half).float().unsqueeze(1)

perm = torch.randperm(samples)
cut = int(samples * 0.8)
train_idx, test_idx = perm[:cut], perm[cut:]
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print('input shape [batch, time, features]:', X_train.shape)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence_output, (hidden, cell) = self.lstm(x)
        # hidden[-1] is the final hidden state from the last LSTM layer.
        return self.output(hidden[-1])

model = LSTMClassifier()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model

In [ ]:
for epoch in range(151):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 30 == 0:
        print(f'epoch={epoch:3d} loss={loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_test))
    preds = (probs >= 0.5).float()
    accuracy = (preds == y_test).float().mean().item()
print(f'test accuracy: {accuracy:.3f}')

## The LSTM idea in plain English

At each time step an LSTM uses learned **gates** to control its memory:

- **forget gate** — how much previous cell-state information to keep;
- **input gate** — how much new information to write;
- **output gate** — how much memory to expose as the current hidden state.

That makes LSTMs useful for ordered data such as time series, sensor readings and text. For many modern NLP tasks Transformers are now more common, but LSTMs remain valuable for understanding sequence modelling and can still be appropriate for smaller sequential problems.